In [1]:
from pathlib import Path
import json

candidates = []

for config_path in Path("/kaggle/input").rglob("run_config.json"):
    config = json.loads(config_path.read_text())

    if (
        config.get("model") == "Qwen/Qwen2.5-0.5B-Instruct"
        and config.get("training_rows") == 10590
        and config.get("seed") == 42
        and (config_path.parent / "best_adapter").is_dir()
    ):
        candidates.append(config_path.parent)

print("Matching baseline runs:")
for path in candidates:
    print(path)

assert len(candidates) == 1, (
    "Attach the saved HealthVer baseline notebook outputs. "
    "Exactly one matching baseline run is needed."
)

BASELINE_RUN_DIR = candidates[0]
BASELINE_ROOT = BASELINE_RUN_DIR.parent

required_paths = {
    "trained_baseline": BASELINE_RUN_DIR / "best_adapter",
    "initial_adapter": BASELINE_ROOT / "initial_adapter_seed42",
    "tokenized_train": BASELINE_ROOT / "tokenized_train",
    "tokenized_validation": BASELINE_ROOT / "tokenized_validation",
    "original_data": BASELINE_ROOT / "original_data",
    "input_config": BASELINE_ROOT / "input_config.json",
    "dataset_metadata": BASELINE_ROOT / "dataset_metadata.json",
}

for name, path in required_paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")
    assert path.exists(), f"Required baseline artifact missing: {path}"

OUTPUT_DIR = Path("/kaggle/working/HealthVer_Random_Experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nBaseline files located. No training or evaluation performed.")

Matching baseline runs:
/kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments/Qwen_LoRA_full_seed42
trained_baseline: FOUND
initial_adapter: FOUND
tokenized_train: FOUND
tokenized_validation: FOUND
original_data: FOUND
input_config: FOUND
dataset_metadata: FOUND

Baseline files located. No training or evaluation performed.


In [2]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import hashlib
import json
import time

import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
)
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

# Restore the exact dataset/model settings from the baseline notebook.
input_config = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
dataset_metadata = json.loads(
    (BASELINE_ROOT / "dataset_metadata.json").read_text()
)

MODEL_ID = input_config["model"]
MODEL_REVISION = input_config["model_revision"]
DATASET_REVISION = dataset_metadata["revision"]
SYSTEM_PROMPT = input_config["system_prompt"]
LABELS = input_config["labels"]
DEVICE = torch.device("cuda:0")
EVAL_BATCH_SIZE = 8

assert torch.cuda.is_available(), "Enable a GPU."

test_path = (
    BASELINE_ROOT / "original_data"
    / f"{DATASET_REVISION}_healthver_test.csv"
)

# Verify that the saved source file is unchanged.
assert hashlib.sha256(test_path.read_bytes()).hexdigest() == (
    dataset_metadata["files"]["test"]["sha256"]
)

test_df = pd.read_csv(test_path, dtype=str, keep_default_na=False)
assert len(test_df) == 1823
assert test_df["id"].is_unique
assert set(test_df["label"]).issubset(LABELS)

tokenizer = AutoTokenizer.from_pretrained(
    str(BASELINE_RUN_DIR / "best_adapter")
)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to(DEVICE)

model = PeftModel.from_pretrained(
    base_model,
    str(BASELINE_RUN_DIR / "best_adapter"),
    is_trainable=False,
)
model.eval()
assert not any(p.requires_grad for p in model.parameters())

# Same decoding configuration used for baseline validation.
eval_generation = GenerationConfig.from_pretrained(
    str(BASELINE_ROOT / "Qwen_ZeroShot_Validation")
)

def make_messages(claim, evidence):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

label_lookup = {label.lower(): label for label in LABELS}

def parse_prediction(text):
    normalized = " ".join(text.strip().lower().split())
    return label_lookup.get(normalized, "INVALID")

TEST_DIR = OUTPUT_DIR / "full_baseline_test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

records = []
maximum_prompt_tokens = 0
started = time.perf_counter()

with torch.inference_mode(), (
    TEST_DIR / "predictions.jsonl"
).open("w") as output:

    for offset in tqdm(
        range(0, len(test_df), EVAL_BATCH_SIZE),
        desc="Full baseline — test",
    ):
        rows = test_df.iloc[offset:offset + EVAL_BATCH_SIZE]

        prompts = [
            tokenizer.apply_chat_template(
                make_messages(claim, evidence),
                tokenize=False,
                add_generation_prompt=True,
            )
            for claim, evidence in zip(rows["claim"], rows["evidence"])
        ]

        inputs = tokenizer(
            prompts,
            padding=True,
            truncation=False,
            add_special_tokens=False,
            return_tensors="pt",
        ).to(DEVICE)

        width = inputs["input_ids"].shape[1]
        maximum_prompt_tokens = max(maximum_prompt_tokens, width)

        # Preserve complete test inputs, within the model's context capacity.
        assert (
            width + eval_generation.max_new_tokens
            <= model.config.max_position_embeddings
        )

        generated = model.generate(
            **inputs,
            generation_config=eval_generation,
        )
        responses = tokenizer.batch_decode(
            generated[:, width:],
            skip_special_tokens=True,
        )

        for uid, gold, response in zip(
            rows["id"], rows["label"], responses
        ):
            record = {
                "id": uid,
                "true_label": gold,
                "predicted_label": parse_prediction(response),
                "raw_response": response,
            }
            records.append(record)
            output.write(json.dumps(record) + "\n")

        del inputs, generated

predictions = pd.DataFrame(records)
assert predictions["id"].tolist() == test_df["id"].tolist()

gold = predictions["true_label"]
predicted = predictions["predicted_label"]

metrics = {
    "run": "full_training_baseline_seed42",
    "evaluation_split": "test",
    "n_examples": len(predictions),
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "adapter_path": str(BASELINE_RUN_DIR / "best_adapter"),
    "checkpoint_selection": "highest validation macro-F1",
    "accuracy": float(accuracy_score(gold, predicted)),
    "macro_f1": float(f1_score(
        gold, predicted, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "weighted_f1": float(f1_score(
        gold, predicted, labels=LABELS,
        average="weighted", zero_division=0,
    )),
    "invalid_rate": float((predicted == "INVALID").mean()),
    "maximum_prompt_tokens": maximum_prompt_tokens,
    "truncation": False,
    "elapsed_minutes": (time.perf_counter() - started) / 60,
}

report = classification_report(
    gold, predicted, labels=LABELS,
    output_dict=True, zero_division=0,
)
matrix_labels = LABELS + ["INVALID"]
matrix = pd.DataFrame(
    confusion_matrix(gold, predicted, labels=matrix_labels),
    index=matrix_labels,
    columns=matrix_labels,
).loc[LABELS]

(TEST_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(TEST_DIR / "classification_report.json").write_text(
    json.dumps(report, indent=2)
)
matrix.to_csv(TEST_DIR / "confusion_matrix.csv")
eval_generation.save_pretrained(TEST_DIR)
(TEST_DIR / "input_config.json").write_text(
    json.dumps(input_config, indent=2)
)

print("\nFULL BASELINE — TEST RESULTS")
print(json.dumps(metrics, indent=2))
print(classification_report(
    gold, predicted, labels=LABELS, digits=4, zero_division=0,
))
print("CONFUSION MATRIX — rows: actual; columns: predicted")
display(matrix)
print("Saved:", TEST_DIR)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Full baseline — test:   0%|          | 0/228 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



FULL BASELINE — TEST RESULTS
{
  "run": "full_training_baseline_seed42",
  "evaluation_split": "test",
  "n_examples": 1823,
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "model_revision": "7ae557604adf67be50417f59c2c2f167def9a775",
  "dataset_revision": "b20ac99ceed62f5264a319fa25a854df1668d85b",
  "adapter_path": "/kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments/Qwen_LoRA_full_seed42/best_adapter",
  "checkpoint_selection": "highest validation macro-F1",
  "accuracy": 0.8008776741634668,
  "macro_f1": 0.7916225867719905,
  "weighted_f1": 0.8017510628030883,
  "invalid_rate": 0.0,
  "maximum_prompt_tokens": 294,
  "truncation": false,
  "elapsed_minutes": 1.187875187116667
}
              precision    recall  f1-score   support

    Supports     0.8012    0.7988    0.8000       671
     Refutes     0.7064    0.7529    0.7289       425
     Neutral     0.8616    0.8308    0.8459       727

    accuracy                         0.8009      1823
   macro avg     0.789

,Supports,Refutes,Neutral,INVALID
Supports,536,82,53,0
Refutes,61,320,44,0
Neutral,72,51,604,0


Saved: /kaggle/working/HealthVer_Random_Experiments/full_baseline_test


In [4]:
import hashlib
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SELECTION_SEED = 42

train_path = (
    BASELINE_ROOT / "original_data"
    / f"{DATASET_REVISION}_healthver_train.csv"
)

assert hashlib.sha256(train_path.read_bytes()).hexdigest() == (
    dataset_metadata["files"]["train"]["sha256"]
)

train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False)
assert len(train_df) == 10590
assert train_df["id"].is_unique

SUBSET_DIR = OUTPUT_DIR / "stratified_random_seed42"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# Explicit row counts: round fractional rows to the nearest integer.
subset_sizes = {
    50: 5295,
    25: 2648,
    10: 1059,
    5: 530,
}

all_indices = np.arange(len(train_df))
subset_indices = {}
summary_rows = []

for percentage, size in subset_sizes.items():
    selected, _ = train_test_split(
        all_indices,
        train_size=size,
        stratify=train_df["label"].to_numpy(),
        random_state=SELECTION_SEED,
    )

    selected = np.sort(selected)
    subset_indices[percentage] = selected.tolist()
    subset = train_df.iloc[selected]

    assert len(subset) == size
    assert subset["id"].is_unique
    assert set(subset["label"]) == set(LABELS)

    manifest = {
        "method": "stratified_random",
        "selection_seed": SELECTION_SEED,
        "dataset_revision": DATASET_REVISION,
        "source_split": "train",
        "source_rows": len(train_df),
        "requested_percentage": percentage,
        "actual_rows": size,
        "actual_percentage": 100 * size / len(train_df),
        "sampling_unit": "row",
        "row_indices": selected.tolist(),
        "ids": subset["id"].tolist(),
        "class_counts": {
            label: int((subset["label"] == label).sum())
            for label in LABELS
        },
    }

    manifest_path = SUBSET_DIR / f"subset_{percentage}pct.json"

    # Avoid silently replacing an existing selection.
    if manifest_path.exists():
        assert json.loads(manifest_path.read_text()) == manifest
    else:
        manifest_path.write_text(json.dumps(manifest, indent=2))

    for label in LABELS:
        count = manifest["class_counts"][label]
        summary_rows.append({
            "subset": f"{percentage}%",
            "total_rows": size,
            "label": label,
            "count": count,
            "subset_class_pct": round(100 * count / size, 2),
            "full_train_class_pct": round(
                100 * (train_df["label"] == label).mean(), 2
            ),
        })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(SUBSET_DIR / "class_distribution.csv", index=False)

print("Saved selections:", SUBSET_DIR)
print("Each subset was sampled independently; nesting is not guaranteed.")
print("No training performed. Validation and test sets unchanged.")

,subset,total_rows,label,count,subset_class_pct,full_train_class_pct
0,50%,5295,Supports,1891,35.71,35.71
1,50%,5295,Refutes,1206,22.78,22.77
2,50%,5295,Neutral,2198,41.51,41.52
3,25%,2648,Supports,946,35.73,35.71
4,25%,2648,Refutes,603,22.77,22.77
5,25%,2648,Neutral,1099,41.50,41.52
6,10%,1059,Supports,378,35.69,35.71
7,10%,1059,Refutes,241,22.76,22.77
8,10%,1059,Neutral,440,41.55,41.52
9,5%,530,Supports,189,35.66,35.71


Saved selections: /kaggle/working/HealthVer_Random_Experiments/stratified_random_seed42
Each subset was sampled independently; nesting is not guaranteed.
No training performed. Validation and test sets unchanged.


In [5]:
import gc
import json
import math
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
    Trainer, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

# ============================================================
# Change these two settings for future selection methods.
# Their manifests must contain row_indices, ids, actual_rows,
# method, selection_seed, and dataset_revision.
# ============================================================
METHOD = "stratified_random"
SELECTION_DIR = OUTPUT_DIR / "stratified_random_seed42"
PERCENTAGES = [50, 25, 10, 5]

DEVICE = torch.device("cuda:0")
EVAL_BATCH_SIZE = 8

input_config = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
dataset_metadata = json.loads(
    (BASELINE_ROOT / "dataset_metadata.json").read_text()
)
full_config = json.loads(
    (BASELINE_RUN_DIR / "run_config.json").read_text()
)

MODEL_ID = input_config["model"]
MODEL_REVISION = input_config["model_revision"]
DATASET_REVISION = dataset_metadata["revision"]
LABELS = input_config["labels"]
SYSTEM_PROMPT = input_config["system_prompt"]

TRAINING_SEED = full_config["seed"]
MICRO_BATCH = full_config["micro_batch_size"]
GRAD_ACCUM = full_config["gradient_accumulation"]
EPOCHS = full_config["epochs"]
LEARNING_RATE = full_config["learning_rate"]

assert torch.cuda.is_available()

# Release models left in memory by earlier cells.
for name in ["trainer", "model", "base_model"]:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

full_train = load_from_disk(str(BASELINE_ROOT / "tokenized_train"))
validation_tokens = load_from_disk(
    str(BASELINE_ROOT / "tokenized_validation")
)

def load_original(split, filename):
    path = BASELINE_ROOT / "original_data" / (
        f"{DATASET_REVISION}_{filename}"
    )
    assert hashlib.sha256(path.read_bytes()).hexdigest() == (
        dataset_metadata["files"][split]["sha256"]
    )
    return pd.read_csv(path, dtype=str, keep_default_na=False)

validation_df = load_original("validation", "healthver_dev.csv")
test_df = load_original("test", "healthver_test.csv")

assert len(validation_df) == 1917
assert len(test_df) == 1823

tokenizer = AutoTokenizer.from_pretrained(
    str(BASELINE_RUN_DIR / "best_adapter")
)
eval_generation = GenerationConfig.from_pretrained(
    str(BASELINE_ROOT / "Qwen_ZeroShot_Validation")
)

full_test_metrics = json.loads(
    (OUTPUT_DIR / "full_baseline_test" / "metrics.json").read_text()
)

def save_json(path, value):
    """Write completed JSON files atomically."""
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

def make_messages(claim, evidence):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

label_lookup = {label.lower(): label for label in LABELS}

def parse_prediction(text):
    return label_lookup.get(
        " ".join(text.strip().lower().split()), "INVALID"
    )

def training_collator(rows):
    padding_values = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    batch = {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding_values.items()
    }
    assert batch["input_ids"].shape[1] <= input_config["max_sequence_length"]
    return batch

# The same generated-label evaluator is used for validation and test.
def evaluate_model(model, frame, folder, split, run_fingerprint):
    folder.mkdir(parents=True, exist_ok=True)
    previous_training = model.training
    previous_padding = tokenizer.padding_side
    records = []
    started = time.perf_counter()

    try:
        model.eval()
        tokenizer.padding_side = "left"

        with torch.inference_mode(), (
            folder / "predictions.jsonl"
        ).open("w") as output:

            for offset in range(0, len(frame), EVAL_BATCH_SIZE):
                rows = frame.iloc[offset:offset + EVAL_BATCH_SIZE]
                prompts = [
                    tokenizer.apply_chat_template(
                        make_messages(claim, evidence),
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for claim, evidence in zip(
                        rows["claim"], rows["evidence"]
                    )
                ]
                inputs = tokenizer(
                    prompts,
                    padding=True,
                    truncation=False,
                    add_special_tokens=False,
                    return_tensors="pt",
                ).to(DEVICE)

                width = inputs["input_ids"].shape[1]
                assert (
                    width + eval_generation.max_new_tokens
                    <= model.config.max_position_embeddings
                )

                generated = model.generate(
                    **inputs, generation_config=eval_generation
                )
                responses = tokenizer.batch_decode(
                    generated[:, width:], skip_special_tokens=True
                )

                for uid, gold, response in zip(
                    rows["id"], rows["label"], responses
                ):
                    record = {
                        "id": uid,
                        "true_label": gold,
                        "predicted_label": parse_prediction(response),
                        "raw_response": response,
                    }
                    records.append(record)
                    output.write(json.dumps(record) + "\n")

                del inputs, generated

        assert [row["id"] for row in records] == frame["id"].tolist()
        gold = [row["true_label"] for row in records]
        predicted = [row["predicted_label"] for row in records]

        metrics = {
            "evaluation_split": split,
            "n_examples": len(records),
            "model": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "dataset_revision": DATASET_REVISION,
            "run_config_sha256": run_fingerprint,
            "accuracy": float(accuracy_score(gold, predicted)),
            "macro_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="macro", zero_division=0,
            )),
            "weighted_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="weighted", zero_division=0,
            )),
            "invalid_rate": float(
                np.mean(np.asarray(predicted) == "INVALID")
            ),
            "elapsed_seconds": time.perf_counter() - started,
        }

        save_json(
            folder / "classification_report.json",
            classification_report(
                gold, predicted, labels=LABELS,
                output_dict=True, zero_division=0,
            ),
        )

        matrix_labels = LABELS + ["INVALID"]
        pd.DataFrame(
            confusion_matrix(gold, predicted, labels=matrix_labels),
            index=matrix_labels,
            columns=matrix_labels,
        ).loc[LABELS].to_csv(folder / "confusion_matrix.csv")

        # Written last, after predictions and reports are complete.
        save_json(folder / "metrics.json", metrics)
        return metrics

    finally:
        tokenizer.padding_side = previous_padding
        model.train(previous_training)
        gc.collect()
        with torch.cuda.device(DEVICE):
            torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class HealthVerTrainer(Trainer):
    def evaluate(
        self, eval_dataset=None, ignore_keys=None,
        metric_key_prefix="eval", **kwargs
    ):
        result = evaluate_model(
            self.model,
            validation_df,
            self.run_directory / "validation"
            / f"step_{self.state.global_step}",
            "validation",
            self.run_fingerprint,
        )

        metrics = {
            f"{metric_key_prefix}_{key}": result[key]
            for key in [
                "accuracy", "macro_f1", "weighted_f1", "invalid_rate"
            ]
        }
        metrics[f"{metric_key_prefix}_runtime"] = result["elapsed_seconds"]

        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(
            self.args, self.state, self.control, metrics
        )
        return metrics

def run_one_percentage(percentage):
    manifest_path = SELECTION_DIR / f"subset_{percentage}pct.json"
    manifest = json.loads(manifest_path.read_text())

    assert manifest["method"] == METHOD
    assert manifest["dataset_revision"] == DATASET_REVISION
    assert len(set(manifest["row_indices"])) == manifest["actual_rows"]

    selected_train = full_train.select(manifest["row_indices"])
    assert len(selected_train) == manifest["actual_rows"]
    assert list(selected_train["id"]) == manifest["ids"]

    run_dir = OUTPUT_DIR / (
        f"{METHOD}_{percentage}pct_seed{TRAINING_SEED}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    # Compatible with the earlier standalone 50% training cell.
    run_config = {
        **full_config,
        "training_rows": len(selected_train),
        "method": METHOD,
        "requested_percentage": percentage,
        "selection_seed": manifest["selection_seed"],
        "selection_sha256": hashlib.sha256(
            manifest_path.read_bytes()
        ).hexdigest(),
    }

    config_path = run_dir / "run_config.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, (
            f"Configuration mismatch in {run_dir}"
        )
    else:
        save_json(config_path, run_config)

    fingerprint = hashlib.sha256(
        json.dumps(run_config, sort_keys=True).encode()
    ).hexdigest()

    state_path = run_dir / "trainer_state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {}

    training_complete = (
        (run_dir / "best_adapter" / "adapter_config.json").exists()
        and (state.get("epoch") or 0) >= EPOCHS - 1e-6
    )
    test_metrics_path = run_dir / "test" / "metrics.json"

    print(f"\n{'=' * 55}")
    print(f"{METHOD}: {percentage}% | {len(selected_train):,} training rows")

    # No GPU work needed when both training and testing are complete.
    if training_complete and test_metrics_path.exists():
        test_metrics = json.loads(test_metrics_path.read_text())
        assert test_metrics["run_config_sha256"] == fingerprint
        print("Completed run found; reusing saved results.")
        return run_dir, state, test_metrics, len(selected_train)

    local_model = local_base = local_trainer = None

    try:
        set_seed(TRAINING_SEED)
        tokenizer.padding_side = "right"

        local_base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION,
            dtype=torch.float16,
            attn_implementation="sdpa",
        ).to(DEVICE)

        adapter_path = (
            run_dir / "best_adapter"
            if training_complete
            else BASELINE_ROOT / "initial_adapter_seed42"
        )
        local_model = PeftModel.from_pretrained(
            local_base,
            str(adapter_path),
            is_trainable=not training_complete,
        )

        local_model.config.use_cache = False
        local_model.config.pad_token_id = tokenizer.pad_token_id
        local_model.gradient_checkpointing_disable()

        if not training_complete:
            for parameter in local_model.parameters():
                if parameter.requires_grad:
                    parameter.data = parameter.data.float()
            del parameter

            steps_per_epoch = math.ceil(
                len(selected_train) / (MICRO_BATCH * GRAD_ACCUM)
            )

            args = SingleGPUArguments(
                output_dir=str(run_dir),
                num_train_epochs=EPOCHS,
                per_device_train_batch_size=MICRO_BATCH,
                gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=LEARNING_RATE,
                warmup_steps=math.ceil(
                    0.03 * EPOCHS * steps_per_epoch
                ),
                lr_scheduler_type="linear",
                optim="adamw_torch",
                weight_decay=0.01,
                max_grad_norm=1.0,
                fp16=True,
                bf16=False,
                gradient_checkpointing=False,
                eval_strategy="epoch",
                save_strategy="epoch",
                save_total_limit=2,
                save_only_model=False,
                load_best_model_at_end=True,
                metric_for_best_model="macro_f1",
                greater_is_better=True,
                logging_steps=25,
                logging_first_step=True,
                report_to="none",
                dataloader_num_workers=0,
                dataloader_pin_memory=True,
                seed=TRAINING_SEED,
                data_seed=TRAINING_SEED,
                label_names=["labels"],
            )

            assert args.world_size == 1
            assert args.device == DEVICE
            assert args.train_batch_size == MICRO_BATCH

            local_trainer = HealthVerTrainer(
                model=local_model,
                args=args,
                train_dataset=selected_train.select_columns(
                    ["input_ids", "attention_mask", "labels"]
                ),
                eval_dataset=validation_tokens,
                data_collator=training_collator,
                processing_class=tokenizer,
            )
            local_trainer.run_directory = run_dir
            local_trainer.run_fingerprint = fingerprint

            checkpoint = get_last_checkpoint(str(run_dir))
            print("Resuming:", checkpoint or "initial untrained adapter")

            torch.cuda.reset_peak_memory_stats(DEVICE)
            result = local_trainer.train(
                resume_from_checkpoint=checkpoint
            )

            # Best validation checkpoint is loaded automatically.
            local_trainer.save_model(str(run_dir / "best_adapter"))
            tokenizer.save_pretrained(run_dir / "best_adapter")
            local_trainer.save_metrics("train", result.metrics)
            local_trainer.save_state()

            save_json(run_dir / "resources.json", {
                "resumed_from": checkpoint,
                "train_call_runtime_seconds":
                    result.metrics.get("train_runtime"),
                "peak_allocated_gib":
                    torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
                "note": (
                    "A resumed call's runtime excludes earlier sessions. "
                    "Selection cost is recorded separately."
                ),
            })

            local_model = local_trainer.model
            state = json.loads(state_path.read_text())
        else:
            print("Training already complete; loaded saved best adapter.")

        # Test results never determine checkpoint selection.
        test_metrics = evaluate_model(
            local_model, test_df, run_dir / "test",
            "test", fingerprint,
        )

        print(
            f"Test accuracy: {test_metrics['accuracy']:.4f} | "
            f"macro-F1: {test_metrics['macro_f1']:.4f}"
        )
        return run_dir, state, test_metrics, len(selected_train)

    finally:
        local_trainer = local_model = local_base = None
        gc.collect()
        with torch.cuda.device(DEVICE):
            torch.cuda.empty_cache()

# Save a cumulative comparison after every completed percentage.
summary_rows = [{
    "method": "full_training",
    "percentage": 100,
    "training_rows": len(full_train),
    "training_seed": TRAINING_SEED,
    "selection_seed": None,
    "best_validation_macro_f1": json.loads(
        (BASELINE_RUN_DIR / "trainer_state.json").read_text()
    )["best_metric"],
    "test_accuracy": full_test_metrics["accuracy"],
    "test_macro_f1": full_test_metrics["macro_f1"],
    "test_weighted_f1": full_test_metrics["weighted_f1"],
    "macro_f1_change_pp": 0.0,
}]

summary_path = OUTPUT_DIR / f"{METHOD}_comparison_seed{TRAINING_SEED}.csv"

for percentage in PERCENTAGES:
    run_dir, state, test_metrics, n_rows = run_one_percentage(percentage)
    manifest = json.loads(
        (SELECTION_DIR / f"subset_{percentage}pct.json").read_text()
    )

    summary_rows.append({
        "method": METHOD,
        "percentage": percentage,
        "training_rows": n_rows,
        "training_seed": TRAINING_SEED,
        "selection_seed": manifest["selection_seed"],
        "best_validation_macro_f1": state["best_metric"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "macro_f1_change_pp": 100 * (
            test_metrics["macro_f1"] - full_test_metrics["macro_f1"]
        ),
    })

    comparison = pd.DataFrame(summary_rows)
    comparison.to_csv(summary_path, index=False)
    display(comparison.round(4))

print("\nAll requested runs completed.")
print("Comparison saved:", summary_path)


stratified_random: 50% | 5,295 training rows


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming: initial untrained adapter


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.199334,No log,0.712572,0.659884,0.708033,0.000000
2,0.110768,No log,0.776213,0.733819,0.772219,0.000000
3,0.060942,No log,0.793427,0.761541,0.795119,0.000000


Test accuracy: 0.7625 | macro-F1: 0.7502


... (19 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.446526,No log,0.352634,0.243191,0.265735,0.000000
2,0.348992,No log,0.611372,0.478937,0.566129,0.000000
3,0.222161,No log,0.602504,0.552152,0.608064,0.000000


Test accuracy: 0.5727 | macro-F1: 0.5485


,method,percentage,training_rows,training_seed,selection_seed,best_validation_macro_f1,test_accuracy,test_macro_f1,test_weighted_f1,macro_f1_change_pp
0,full_training,100,10590,42,NaN,0.8016,0.8009,0.7916,0.8018,0.0000
1,stratified_random,50,5295,42,42.0,0.7615,0.7625,0.7502,0.7619,-4.1392
2,stratified_random,25,2648,42,42.0,0.6910,0.6939,0.6704,0.6914,-12.1202
3,stratified_random,10,1059,42,42.0,0.6201,0.6347,0.6158,0.6348,-17.5840
4,stratified_random,5,530,42,42.0,0.5522,0.5727,0.5485,0.5744,-24.3108



All requested runs completed.
Comparison saved: /kaggle/working/HealthVer_Random_Experiments/stratified_random_comparison_seed42.csv
